# Detecting protest events on my data

In [1]:
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [2]:
from protest_impact.data.protests.detection import load_glpn_dataset

glpn = load_glpn_dataset()
glpn = glpn.rename_column("excerpt", "text")

Using custom data configuration default-386fb51d30c78e56
Found cached dataset csv (/Users/david/.cache/huggingface/datasets/csv/default-386fb51d30c78e56/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317)


  0%|          | 0/5 [00:00<?, ?it/s]

Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/csv/default-386fb51d30c78e56/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317/cache-1c6c61baf1fc4966.arrow
Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/csv/default-386fb51d30c78e56/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317/cache-bbadab134ac9ce97.arrow
Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/csv/default-386fb51d30c78e56/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317/cache-5b646fe4fc3137d1.arrow
Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/csv/default-386fb51d30c78e56/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317/cache-52989505097dfa79.arrow
Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/csv/default-386fb51d30c78e56/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317

In [3]:
from protest_impact.data.protests.detection import load_aglpn_dataset

protest_news = load_aglpn_dataset()
protest_news, protest_news["train"].features

Using custom data configuration protest-db1d3e4a3a280c30
Found cached dataset json (/Users/david/.cache/huggingface/datasets/json/protest-db1d3e4a3a280c30/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51)


  0%|          | 0/1 [00:00<?, ?it/s]

Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/json/protest-db1d3e4a3a280c30/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-d5126aa63730299d.arrow
Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/json/protest-db1d3e4a3a280c30/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-16ddc248b5353cc0.arrow
Loading cached split indices for dataset at /Users/david/.cache/huggingface/datasets/json/protest-db1d3e4a3a280c30/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-1618ed8fb2794fbb.arrow and /Users/david/.cache/huggingface/datasets/json/protest-db1d3e4a3a280c30/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-55f9fd45111c2ca4.arrow
Loading cached split indices for dataset at /Users/david/.cache/huggingface/datasets/json/protest-db1d3e4a3a280c30/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-a6e4

(DatasetDict({
     train: Dataset({
         features: ['text', 'meta', '_input_hash', '_task_hash', 'spans', 'options', 'accept', '_view_id', 'config', 'answer', '_timestamp', 'label'],
         num_rows: 575
     })
     dev: Dataset({
         features: ['text', 'meta', '_input_hash', '_task_hash', 'spans', 'options', 'accept', '_view_id', 'config', 'answer', '_timestamp', 'label'],
         num_rows: 115
     })
     test: Dataset({
         features: ['text', 'meta', '_input_hash', '_task_hash', 'spans', 'options', 'accept', '_view_id', 'config', 'answer', '_timestamp', 'label'],
         num_rows: 460
     })
 }),
 {'text': Value(dtype='string', id=None),
  'meta': {'date': Value(dtype='string', id=None),
   'title': Value(dtype='string', id=None),
   'url': Value(dtype='string', id=None),
   'homepage': Value(dtype='string', id=None),
   'crawl_engine': Value(dtype='string', id=None),
   'crawl_query': Value(dtype='string', id=None),
   'pattern': Value(dtype='string', id=None)

In [4]:
len(glpn["train"]), len(protest_news["train"])

(1914, 575)

In [10]:
import spacy

from protest_impact.data.protests.config import search_regex

nlp = spacy.load("de_core_news_sm", disable=["parser", "tagger", "ner", "tokenizer"])
nlp.add_pipe("sentencizer")


def kwic(text, n=0):
    sents = list(nlp(text).sents)
    kwics_nrs = set()
    for i, sent in enumerate(sents):
        if search_regex.search(sent.text_with_ws):
            kwics_nrs.add(i)
            for j in range(1, n + 1):
                kwics_nrs.add(i - j)
                kwics_nrs.add(i + j)
    kwic_text = ""
    for kwic_nr in sorted(list(kwics_nrs)):
        if kwic_nr >= 0 and kwic_nr < len(sents):
            if kwic_nr - 1 not in kwics_nrs:
                kwic_text += "\n...\n"
            kwic_text += sents[kwic_nr].text_with_ws
    return kwic_text

In [11]:
def kwic_dataset(dataset, n=0):
    return dataset.map(
        lambda x: {
            "text": x["meta"]["title"]
            + "\n\n"
            + kwic("\n".join(list(x["text"].split("\n"))[1:]), n=n)
        }
    )

In [12]:
protest_news = kwic_dataset(protest_news, n=2)

  0%|          | 0/575 [00:00<?, ?ex/s]

  0%|          | 0/115 [00:00<?, ?ex/s]

  0%|          | 0/460 [00:00<?, ?ex/s]

In [ ]:
if device == "cpu":
    n = 20
    glpn["train"] = glpn["train"].shuffle(seed=20230206).select(range(n))
    glpn["dev"] = glpn["dev"].shuffle(seed=20230206).select(range(n))
    glpn["test"] = glpn["test"].shuffle(seed=20230206).select(range(n))
    glpn["test.time"] = glpn["test.time"].shuffle(seed=20230206).select(range(n))
    glpn["test.loc"] = glpn["test.loc"].shuffle(seed=20230206).select(range(n))

    protest_news["train"] = (
        protest_news["train"].shuffle(seed=20230206).select(range(n))
    )
    protest_news["dev"] = protest_news["dev"].shuffle(seed=20230206).select(range(n))
    protest_news["test"] = protest_news["test"].shuffle(seed=20230206).select(range(n))

In [13]:
import evaluate
import numpy as np

metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [5]:
from pathlib import Path

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)


def train_model(model, tokenizer, description, dataset):
    model_location = Path("models") / description / "model"

    if model_location.exists():
        return AutoModelForSequenceClassification.from_pretrained(model_location)

    def tokenize_function(examples):
        return tokenizer(
            examples["text"], padding="max_length", truncation=True, max_length=512
        )

    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    training_args = TrainingArguments(
        output_dir=model_location.parent / "checkpoints",
        evaluation_strategy="epoch",
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        learning_rate=5e-6,
        weight_decay=0.2,
        num_train_epochs=6,
        fp16=True,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["dev"],
        compute_metrics=compute_metrics,
    )
    trainer.train()
    trainer.save_model(model_location)
    return model

AttributeError: module 'numpy' has no attribute '_no_nep50_warning'

In [16]:
# model_name = "deepset/gelectra-base"
model_name = "deepset/gelectra-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# model_vanilla = AutoModelForSequenceClassification.from_pretrained(model_name).to(
#     device
# )

Some weights of the model checkpoint at deepset/gelectra-large were not used when initializing ElectraForSequenceClassification: ['discriminator_predictions.dense.bias', 'discriminator_predictions.dense.weight', 'discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense_prediction.weight']
- This IS expected if you are initializing ElectraForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at deepset/gelectra-large and are newly initialized: ['classifier.dense.weight', 'classif

In [4]:
model_glpn = train_model(
    model_vanilla, tokenizer, f"{model_name.replace('/', '-')}-glpn", glpn
)

NameError: name 'train_model' is not defined

In [18]:
from evaluate import TextClassificationEvaluator


def evaluate(model, tokenizer, test_set):
    eval_results = TextClassificationEvaluator().compute(
        model_or_pipeline=model,
        data=test_set,
        input_column="text",
        label_column="label",
        label_mapping={"LABEL_0": 0, "LABEL_1": 1},
        # label_mapping={"irrelevant": 0, "relevant": 1},
        # label_mapping={"LABEL_0": "irrelevant", "LABEL_1": "relevant"},
        tokenizer=tokenizer,
        metric=metric,
    )
    return eval_results

In [29]:
from sklearn.metrics import classification_report
from transformers import pipeline


def evaluate_detail(model, tokenizer, test_set):
    classifier = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        device=device,
        padding="max_length",
        truncation=True,
        max_length=512,
    )
    predictions = classifier(a["text"] for a in test_set)
    y_pred = [int(a["label"][-1]) for a in predictions]
    y_true = [a["label"] for a in test_set]
    print(sum(y_true) / len(y_true))
    print(sum(y_pred) / len(y_pred))
    print(classification_report(y_true, y_pred))

    # plot scores vs labels
    import matplotlib.pyplot as plt

    scores_ = [
        (1 - a["score"] if a["label"] == "LABEL_0" else a["score"]) for a in predictions
    ]
    correctness = [ypred == ytrue for ypred, ytrue in zip(y_pred, y_true)]
    plt.scatter(scores_, correctness)

In [21]:
evaluate(model_glpn, tokenizer, glpn["test"])

{'f1': 0.9392592592592591,
 'total_time_in_seconds': 17.051019496982917,
 'samples_per_second': 32.08019321641082,
 'latency_in_seconds': 0.031171882078579374}

In [30]:
evaluate_detail(model_glpn, tokenizer, glpn["test"])

0.603290676416819
0.6307129798903108
              precision    recall  f1-score   support

           0       0.94      0.87      0.90       217
           1       0.92      0.96      0.94       330

    accuracy                           0.93       547
   macro avg       0.93      0.92      0.92       547
weighted avg       0.93      0.93      0.92       547



In [31]:
evaluate(model_glpn, tokenizer, protest_news["test"])

{'f1': 0.5893719806763285,
 'total_time_in_seconds': 11.769163428107277,
 'samples_per_second': 39.08519095770407,
 'latency_in_seconds': 0.025585137887189732}

In [32]:
evaluate_detail(model_glpn, tokenizer, protest_news["test"])

0.13260869565217392
0.3173913043478261
              precision    recall  f1-score   support

           0       1.00      0.79      0.88       399
           1       0.42      1.00      0.59        61

    accuracy                           0.82       460
   macro avg       0.71      0.89      0.74       460
weighted avg       0.92      0.82      0.84       460



In [33]:
model_protest_news = train_model(
    model_vanilla,
    tokenizer,
    f"{model_name.replace('/', '-')}-protest-news",
    protest_news,
)

In [34]:
evaluate(model_protest_news, tokenizer, protest_news["test"])

{'f1': 0.7819548872180451,
 'total_time_in_seconds': 11.883023499045521,
 'samples_per_second': 38.710686723538714,
 'latency_in_seconds': 0.02583265978053374}

In [35]:
evaluate_detail(model_protest_news, tokenizer, protest_news["test"])

0.13260869565217392
0.1565217391304348
              precision    recall  f1-score   support

           0       0.98      0.95      0.96       399
           1       0.72      0.85      0.78        61

    accuracy                           0.94       460
   macro avg       0.85      0.90      0.87       460
weighted avg       0.94      0.94      0.94       460



In [36]:
model_glpn_protest_news = train_model(
    model_glpn,
    tokenizer,
    f"{model_name.replace('/', '-')}-glpn-protest-news",
    protest_news,
)

In [37]:
evaluate(model_glpn_protest_news, tokenizer, protest_news["test"])

{'f1': 0.7647058823529411,
 'total_time_in_seconds': 11.809777463087812,
 'samples_per_second': 38.950776290049355,
 'latency_in_seconds': 0.0256734292675822}

In [38]:
evaluate_detail(model_protest_news, tokenizer, protest_news["test"])

0.13260869565217392
0.1565217391304348
              precision    recall  f1-score   support

           0       0.98      0.95      0.96       399
           1       0.72      0.85      0.78        61

    accuracy                           0.94       460
   macro avg       0.85      0.90      0.87       460
weighted avg       0.94      0.94      0.94       460



Make predictions for annotating more relevant articles: